# Notebook 08 — Portfolio Optimization & Risk Management

**Phase 3 · Strategy Integration (2 / 2)**

---

## 🎯 Learning Objectives

| # | Goal |
|---|------|
| 1 | **Normalize** ensemble weights while respecting a cash floor |
| 2 | Apply **position limits** (max allocation per asset) |
| 3 | Implement regime-dependent **cash floors** (bull 20%, ranging 40%, bear 50%) |
| 4 | Build a two-level **circuit breaker** (L1: reduce, L2: halt) |
| 5 | Understand **half-Kelly** position sizing |
| 6 | Generate **rebalance orders** from current → target weights |
| 7 | Compare with production `PortfolioOptimizer`, `RiskManager`, `CircuitBreaker` |

### Prerequisites
- NB07 (Ensemble output → `target_weights`)

In [ ]:
# ── Setup ──────────────────────────────────────────────────
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = str(pathlib.Path.cwd().resolve().parents[1])
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (12, 5), "axes.grid": True})
print("✅ Imports OK  |  Project root:", ROOT)

---
## 1 · The Risk Pipeline

After the ensemble produces `target_weights`, the portfolio goes through a **risk pipeline** before execution:

```
ensemble_combine()
    │
    ▼
target_weights {sym: raw_weight}  +  cash_allocation
    │
    ├── 1. normalize_weights(cash_floor)
    ├── 2. enforce_position_limit(max_pct=0.10)
    ├── 3. circuit_breaker.evaluate(drawdown)
    │       ├── "ok"     → proceed
    │       ├── "reduce" → halve all weights
    │       └── "halt"   → zero all weights
    │
    ├── 4. Re-normalize after clipping
    │
    └── generate_rebalance_orders(current, target, min_drift=0.15)
```

### Risk Parameters (from `config/strategy_params.yaml`)

```yaml
risk:
  max_position_pct: 0.10
  cash_floor_bull:    0.20
  cash_floor_ranging: 0.40
  cash_floor_bear:    0.50
  circuit_breaker:
    level_one: 0.03   # 3% drawdown → reduce
    level_two: 0.05   # 5% drawdown → halt
```

---
## 2 · Weight Normalization with Cash Floor

In [ ]:
def normalize_weights(weights: dict[str, float], cash_floor: float = 0.0) -> dict[str, float]:
    """Scale positive weights so total = (1 - cash_floor)."""
    positive = {s: max(w, 0.0) for s, w in weights.items() if w > 0}
    total = sum(positive.values())
    if total <= 0 or cash_floor >= 1.0:
        return {}
    investable = 1.0 - max(cash_floor, 0.0)
    return {s: (w / total) * investable for s, w in positive.items()}

# Example: raw ensemble output from NB07
raw_weights = {
    "BTCUSDT":  0.145,
    "ETHUSDT":  0.263,
    "SOLUSDT":  0.225,
    "BNBUSDT":  0.035,
    "XRPUSDT":  0.015,
    "AVAXUSDT": 0.125,
}

print("Raw weights (sum={:.3f}):".format(sum(raw_weights.values())))
for s, w in raw_weights.items():
    print(f"  {s:12s}: {w:.4f}")

# Normalize with bull, ranging, and bear cash floors
cash_floors = {"bull": 0.20, "ranging": 0.40, "bear": 0.50}

print()
for regime, cf in cash_floors.items():
    normed = normalize_weights(raw_weights, cash_floor=cf)
    print(f"{regime.upper()} (cash_floor={cf:.0%}, investable={1-cf:.0%}):")
    for s in sorted(normed, key=normed.get, reverse=True):
        print(f"  {s:12s}: {normed[s]:.4f}")
    print(f"  {'Total':12s}: {sum(normed.values()):.4f}  |  Cash: {1-sum(normed.values()):.4f}\n")

In [ ]:
# Visualize: how the same raw weights look under different cash floors
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
bar_colours = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12", "#9b59b6", "#1abc9c"]

for ax, (regime, cf) in zip(axes, cash_floors.items()):
    normed = normalize_weights(raw_weights, cash_floor=cf)
    syms = sorted(normed.keys())
    vals = [normed[s] for s in syms]
    ax.bar(syms, vals, color=bar_colours[:len(syms)], alpha=0.8)
    ax.bar(["CASH"], [1- sum(vals)], color="#95a5a6", alpha=0.6)
    ax.set_title(f"{regime.upper()} (cash_floor={cf:.0%})")
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylim(0, 0.55)

axes[0].set_ylabel("Weight")
fig.suptitle("Weight Normalization with Regime Cash Floors", fontsize=13)
plt.tight_layout()
plt.show()

---
## 3 · Position Limits

No single asset should exceed **10%** of the portfolio — this is the `max_position_pct` guard.

$$
w_i^{\text{clipped}} = \min(w_i, 0.10)
$$

After clipping, we must **re-normalize** because the total may now be less than the investable fraction.

In [ ]:
def enforce_position_limit(weights: dict[str, float], max_pct: float) -> dict[str, float]:
    """Clip each weight to max_pct."""
    return {s: min(max(w, 0.0), max_pct) for s, w in weights.items() if w > 0}

MAX_POS = 0.10

# Take bull-normalized weights
bull_normed = normalize_weights(raw_weights, cash_floor=0.20)
clipped = enforce_position_limit(bull_normed, MAX_POS)

print("Before clipping → After clipping:")
for s in sorted(bull_normed, key=bull_normed.get, reverse=True):
    before = bull_normed[s]
    after = clipped[s]
    flag = " ✂️ CLIPPED" if after < before else ""
    print(f"  {s:12s}: {before:.4f} → {after:.4f}{flag}")

print(f"\nTotal before: {sum(bull_normed.values()):.4f}")
print(f"Total after:  {sum(clipped.values()):.4f}")
print(f"Lost weight:  {sum(bull_normed.values()) - sum(clipped.values()):.4f}")

In [ ]:
# Re-normalize after clipping to fill the investable fraction
def full_risk_pipeline(raw_weights, cash_floor, max_pos):
    """Normalize → clip → re-normalize."""
    step1 = normalize_weights(raw_weights, cash_floor)
    step2 = enforce_position_limit(step1, max_pos)
    step3 = normalize_weights(step2, cash_floor)  # Re-normalize
    return step1, step2, step3

s1, s2, s3 = full_risk_pipeline(raw_weights, 0.20, 0.10)

print(f"{'Step':6s} {'Sum':>8s}  {'Max':>8s}")
print(f"{'Norm':6s} {sum(s1.values()):8.4f}  {max(s1.values()):8.4f}")
print(f"{'Clip':6s} {sum(s2.values()):8.4f}  {max(s2.values()):8.4f}")
print(f"{'Re-Norm':6s} {sum(s3.values()):8.4f}  {max(s3.values()):8.4f}")

> **Note:** After re-normalization, some weights may again exceed `max_pos` if there are few assets. In production, we iterate clip → re-normalize until all weights are below the limit.

---
## 4 · Circuit Breakers

Circuit breakers monitor **portfolio drawdown** and take protective action:

| Level | Drawdown | Action |
|-------|----------|--------|
| OK | < 3% | Normal trading |
| **L1** | 3% – 5% | `reduce` — halve all position weights |
| **L2** | ≥ 5% | `halt` — zero all weights (full cash) |

In [ ]:
class CircuitBreaker:
    def __init__(self, level_one: float = 0.03, level_two: float = 0.05):
        self.level_one = level_one
        self.level_two = level_two
    
    def evaluate(self, drawdown_pct: float) -> str:
        if drawdown_pct >= self.level_two:
            return "halt"
        if drawdown_pct >= self.level_one:
            return "reduce"
        return "ok"

cb = CircuitBreaker(level_one=0.03, level_two=0.05)

# Sweep drawdown values
drawdowns = np.arange(0, 0.08, 0.005)
actions = [cb.evaluate(dd) for dd in drawdowns]
action_colours = {"ok": "#2ecc71", "reduce": "#f39c12", "halt": "#e74c3c"}

fig, ax = plt.subplots(figsize=(12, 3))
for i, (dd, action) in enumerate(zip(drawdowns, actions)):
    ax.bar(i, 1, color=action_colours[action], width=0.9, alpha=0.7)
    ax.text(i, 0.5, action, ha="center", va="center", fontsize=8, fontweight="bold")

ax.set_xticks(range(len(drawdowns)))
ax.set_xticklabels([f"{dd:.1%}" for dd in drawdowns], rotation=45)
ax.set_xlabel("Portfolio Drawdown")
ax.set_yticks([])
ax.set_title("Circuit Breaker Response")
plt.tight_layout()
plt.show()

In [ ]:
def apply_circuit_breaker(weights: dict[str, float], drawdown: float) -> dict[str, float]:
    """Apply circuit breaker to portfolio weights."""
    action = cb.evaluate(drawdown)
    if action == "halt":
        return {s: 0.0 for s in weights}  # All cash
    elif action == "reduce":
        return {s: w * 0.5 for s, w in weights.items()}  # Halve exposure
    return weights  # No change

# Example: portfolio at different drawdown levels
final_weights = s3  # From full_risk_pipeline above

for dd in [0.01, 0.035, 0.06]:
    adjusted = apply_circuit_breaker(final_weights, dd)
    action = cb.evaluate(dd)
    total = sum(adjusted.values())
    print(f"Drawdown={dd:.1%}  Action={action:6s}  Total Equity={total:.4f}  Cash={1-total:.4f}")

---
## 5 · Simulating Drawdown & Circuit Breaker Impact

In [ ]:
# Simulate an equity curve that triggers circuit breakers
np.random.seed(99)
n_days = 100
daily_returns = np.concatenate([
    np.random.normal(0.002, 0.012, 40),  # Normal period
    np.random.normal(-0.005, 0.020, 30), # Drawdown period
    np.random.normal(0.003, 0.010, 30),  # Recovery
])

# Without circuit breaker
equity_no_cb = [1.0]
for r in daily_returns:
    equity_no_cb.append(equity_no_cb[-1] * (1 + r))
equity_no_cb = np.array(equity_no_cb)

# With circuit breaker
equity_with_cb = [1.0]
cb_actions = ["ok"]
for i, r in enumerate(daily_returns):
    # Calculate current drawdown
    peak = max(equity_with_cb)
    current = equity_with_cb[-1]
    dd = (peak - current) / peak if peak > 0 else 0
    
    action = cb.evaluate(dd)
    cb_actions.append(action)
    
    # Scale return by exposure
    if action == "halt":
        effective_r = 0.0  # All cash
    elif action == "reduce":
        effective_r = r * 0.5
    else:
        effective_r = r
    
    equity_with_cb.append(equity_with_cb[-1] * (1 + effective_r))

equity_with_cb = np.array(equity_with_cb)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={"height_ratios": [3, 1]})

ax1.plot(equity_no_cb, label="No Circuit Breaker", lw=1.5, alpha=0.8, color="#e74c3c")
ax1.plot(equity_with_cb, label="With Circuit Breaker", lw=1.5, alpha=0.8, color="#2ecc71")
ax1.set_ylabel("Equity")
ax1.set_title("Circuit Breaker Impact on Equity Curve")
ax1.legend()

# Drawdown of unprotected portfolio
peak_eq = np.maximum.accumulate(equity_no_cb)
drawdown = (peak_eq - equity_no_cb) / peak_eq
ax2.fill_between(range(len(drawdown)), drawdown, alpha=0.4, color="#e74c3c")
ax2.axhline(0.03, ls="--", color="#f39c12", lw=1, label="L1 (3%)")
ax2.axhline(0.05, ls="--", color="#e74c3c", lw=1, label="L2 (5%)")
ax2.set_ylabel("Drawdown")
ax2.set_xlabel("Day")
ax2.legend(fontsize=8)
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

max_dd_no_cb = drawdown.max()
peak_cb = np.maximum.accumulate(equity_with_cb)
max_dd_with_cb = ((peak_cb - equity_with_cb) / peak_cb).max()
print(f"Max drawdown without CB: {max_dd_no_cb:.2%}")
print(f"Max drawdown with CB:    {max_dd_with_cb:.2%}")
print(f"Final equity without CB: {equity_no_cb[-1]:.4f}")
print(f"Final equity with CB:    {equity_with_cb[-1]:.4f}")

---
## 6 · Half-Kelly Position Sizing

The **Kelly Criterion** tells us the optimal fraction of capital to bet:

$$
f^* = \frac{p}{a} - \frac{q}{b}
$$

where $p$ = win probability, $q = 1-p$, $a$ = average loss, $b$ = average win.

In practice, we use **half-Kelly** ($f^*/2$) because:
- Kelly assumes perfect knowledge of $p$ and $b/a$
- Real distributions have fat tails
- Half-Kelly achieves ~75% of Kelly growth with ~50% of the variance

In [ ]:
def kelly_fraction(win_prob: float, avg_win: float, avg_loss: float) -> float:
    """Full Kelly fraction."""
    q = 1 - win_prob
    if avg_loss == 0 or avg_win == 0:
        return 0.0
    return win_prob / avg_loss - q / avg_win

def half_kelly(win_prob: float, avg_win: float, avg_loss: float) -> float:
    """Half-Kelly: safer, more practical."""
    return max(0.0, kelly_fraction(win_prob, avg_win, avg_loss) / 2)

# Example: our bot historically wins 55% of trades, avg_win=2.5%, avg_loss=1.8%
p = 0.55
avg_w = 0.025
avg_l = 0.018

fk = kelly_fraction(p, avg_w, avg_l)
fhk = half_kelly(p, avg_w, avg_l)
print(f"Win rate:       {p:.0%}")
print(f"Avg win:        {avg_w:.1%}")
print(f"Avg loss:       {avg_l:.1%}")
print(f"Full Kelly:     {fk:.1%}")
print(f"Half Kelly:     {fhk:.1%}")

In [ ]:
# Sensitivity: Kelly fraction vs win rate
win_rates = np.arange(0.40, 0.75, 0.01)
full_kellys = [kelly_fraction(p, avg_w, avg_l) for p in win_rates]
half_kellys = [half_kelly(p, avg_w, avg_l) for p in win_rates]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(win_rates, full_kellys, label="Full Kelly", lw=2, color="#e74c3c")
ax.plot(win_rates, half_kellys, label="Half Kelly", lw=2, color="#2ecc71")
ax.axhline(0.10, ls="--", color="gray", lw=0.8, label="Max Position (10%)")
ax.axhline(0, ls="-", color="black", lw=0.5)
ax.axvline(0.55, ls=":", color="#3498db", lw=1, label="Our Win Rate (55%)")
ax.set_xlabel("Win Rate")
ax.set_ylabel("Optimal Fraction")
ax.set_title("Kelly Criterion vs Win Rate")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7 · Rebalance Order Generation

Once we have final target weights, we compare them against **current** portfolio weights and generate orders. Only changes above `min_drift=0.15` (i.e., 15 percentage points) trigger trades.

In [ ]:
def generate_rebalance_orders(
    current: dict[str, float],
    target: dict[str, float],
    min_drift: float = 0.0,
) -> list[dict]:
    """Generate BUY/SELL proposals for material weight differences."""
    orders = []
    all_symbols = set(current) | set(target)
    for sym in sorted(all_symbols):
        t = target.get(sym, 0.0)
        c = current.get(sym, 0.0)
        drift = t - c
        if abs(drift) <= min_drift:
            continue
        orders.append({
            "side": "BUY" if drift > 0 else "SELL",
            "symbol": sym,
            "target_weight": t,
            "current_weight": c,
            "drift": drift,
        })
    return orders

# Current portfolio (from last rebalance)
current_weights = {
    "BTCUSDT": 0.10,
    "ETHUSDT": 0.10,
    "SOLUSDT": 0.05,
    "BNBUSDT": 0.10,
    "XRPUSDT": 0.10,
    "AVAXUSDT": 0.05,
}

# New target (after full risk pipeline)
target = s3  # From Section 3

# Generate orders
MIN_DRIFT = 0.02  # 2% threshold for demo
orders = generate_rebalance_orders(current_weights, target, min_drift=MIN_DRIFT)

print(f"Rebalance orders (min_drift={MIN_DRIFT:.0%}):")
print(f"{'Side':<5} {'Symbol':<12} {'Current':>8} {'Target':>8} {'Drift':>8}")
print("-" * 45)
for o in orders:
    print(f"{o['side']:<5} {o['symbol']:<12} {o['current_weight']:8.4f} {o['target_weight']:8.4f} {o['drift']:+8.4f}")

In [ ]:
# Visualize current vs target
all_syms = sorted(set(current_weights) | set(target))
x = np.arange(len(all_syms))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w/2, [current_weights.get(s, 0) for s in all_syms], w, label="Current", color="#3498db", alpha=0.7)
ax.bar(x + w/2, [target.get(s, 0) for s in all_syms], w, label="Target", color="#e74c3c", alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(all_syms, rotation=45)
ax.set_ylabel("Weight")
ax.set_title("Current vs Target Portfolio Weights")
ax.legend()
ax.axhline(0.10, ls="--", color="gray", lw=0.8, label="Max Position")
plt.tight_layout()
plt.show()

---
## 8 · Production Code Comparison

In [ ]:
from bot.strategy.portfolio_optimizer import PortfolioOptimizer, normalize_weights as prod_normalize
from bot.risk.risk_manager import RiskManager, enforce_position_limit as prod_enforce
from bot.risk.circuit_breaker import CircuitBreaker as ProdCB
from bot.execution.order_executor import generate_rebalance_orders as prod_gen_orders

# 1. Normalization
po = PortfolioOptimizer(cash_floor=0.20)
prod_normed = po.optimize(raw_weights)
our_normed = normalize_weights(raw_weights, 0.20)
match_norm = all(abs(prod_normed.get(s, 0) - our_normed.get(s, 0)) < 1e-9 for s in set(prod_normed) | set(our_normed))
print(f"normalize_weights match: {'✅' if match_norm else '❌'}")

# 2. Position limits
rm = RiskManager(max_position_pct=0.10)
prod_clipped = rm.apply_position_limits(prod_normed)
our_clipped = enforce_position_limit(our_normed, 0.10)
match_clip = all(abs(prod_clipped.get(s, 0) - our_clipped.get(s, 0)) < 1e-9 for s in set(prod_clipped) | set(our_clipped))
print(f"enforce_position_limit match: {'✅' if match_clip else '❌'}")

# 3. Circuit breaker
pcb = ProdCB(level_one=0.03, level_two=0.05)
for dd in [0.01, 0.035, 0.06]:
    ours = cb.evaluate(dd)
    prod = pcb.evaluate(dd)
    print(f"CB dd={dd:.1%}: ours={ours:6s} prod={prod:6s} {'✅' if ours == prod else '❌'}")

---
## 9 · Full Risk Pipeline Summary

```
ensemble target_weights
        │
        ▼
   ┌────────────────────────┐
   │ normalize_weights()    │ → sum = 1 - cash_floor
   │ (cash_floor by regime) │
   └────────┬───────────────┘
            ▼
   ┌────────────────────────┐
   │ enforce_position_limit │ → clip each to 10%
   │ (max_position=0.10)    │
   └────────┬───────────────┘
            ▼
   ┌────────────────────────┐
   │ circuit_breaker.eval() │ → ok / reduce / halt
   │ (L1=3%, L2=5%)        │
   └────────┬───────────────┘
            ▼
   ┌────────────────────────┐
   │ re-normalize           │ → ensure sum correct
   └────────┬───────────────┘
            ▼
   ┌────────────────────────┐
   │ generate_rebalance_    │ → BUY / SELL orders
   │ orders(min_drift=0.15) │
   └────────────────────────┘
```

---
## 🔬 Exercises

1. **Iterative Clipping:** Implement a loop that clips → re-normalizes until all weights ≤ `max_pos`. How many iterations does it take with 3 assets?

2. **Dynamic Circuit Breaker:** Modify L1/L2 thresholds based on regime: bull (4%/7%), ranging (3%/5%), bear (2%/3%). How does this affect drawdown?

3. **Kelly + Regime:** Compute Kelly fractions per-strategy per-regime using simulated win rates. Which regime-strategy pair has the highest Kelly fraction?

4. **Transaction Costs:** Add a 0.1% fee per trade to the rebalance order generator. What minimum drift becomes optimal to minimize cost drag?

---
## ✅ Knowledge Check

1. Why do we normalize weights *before* applying position limits?
2. What happens to unllocated weight after position clipping? Where does it go?
3. Why is L2 (halt) more aggressive than L1 (reduce)? When would you reset from L2?
4. Why half-Kelly instead of full Kelly?
5. What is `min_drift` and why does it exist?

---
## 🔗 Next

**[NB09 — Backtesting Engine →](09_Backtesting_Engine.ipynb)**

We'll build a walk-forward backtester to test the entire pipeline on historical data, computing Sharpe, Sortino, Calmar, profit factor, and win rate.